# NB13 — Deployment Preparation & Export

This notebook compiles the final trained artifacts and prepares them for deployment on Hugging Face Spaces. 

**Steps performed:**
1. Train and save the Ensemble Stacker model (`lr_stacker.joblib`).
2. Aggregate all artifacts (`knowledge_base.json`, `bandit_state.json`, `lr_stacker.joblib`) into the `models/ensemble` folder.
3. Provide the commands to upload the models to the Hugging Face Hub (as per the Revised Deployment Plan).


In [1]:
import numpy as np
import joblib
import json
import shutil
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("Loading validation probabilities to train LR stacker...")

# Load validation probabilities
roberta_probs = np.load('../models/roberta_distilled/val_probs.npy')
modernbert_probs = np.load('../models/modernbert_full/val_probs.npy')
val_labels = np.load('../models/roberta_distilled/val_labels.npy')

X_val = np.hstack([roberta_probs, modernbert_probs])
y_val = val_labels

print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

# Train the stacker
print("Training Logistic Regression stacker...")
stacker = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])
stacker.fit(X_val, y_val)

# Save the stacker
os.makedirs('../models/ensemble', exist_ok=True)
joblib.dump(stacker, '../models/ensemble/lr_stacker.joblib')
print("✅ Saved Stacked Model to ../models/ensemble/lr_stacker.joblib")


Loading validation probabilities to train LR stacker...
X_val shape: (331178, 20)
y_val shape: (331178,)
Training Logistic Regression stacker...


C:\Users\nwagb\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


✅ Saved Stacked Model to ../models/ensemble/lr_stacker.joblib


In [2]:
print("Gathering artifacts for deployment...")

# Copy knowledge base
kb_src = '../models/rag_pipeline/knowledge_base.json'
kb_dst = '../models/ensemble/knowledge_base.json'
if os.path.exists(kb_src):
    shutil.copy(kb_src, kb_dst)
    print("✅ Copied knowledge_base.json to ../models/ensemble/")
else:
    print(f"⚠️ {kb_src} not found!")

# Copy bandit state
bandit_src = '../models/rl_routing/bandit_state.json'
bandit_dst = '../models/ensemble/bandit_state.json'
if os.path.exists(bandit_src):
    shutil.copy(bandit_src, bandit_dst)
    print("✅ Copied bandit_state.json to ../models/ensemble/")
else:
    print(f"⚠️ {bandit_src} not found!")


Gathering artifacts for deployment...


✅ Copied knowledge_base.json to ../models/ensemble/
✅ Copied bandit_state.json to ../models/ensemble/


### Upload to Hugging Face Hub

Run the cell below to push your models to the Hugging Face Hub. 
**Make sure you have logged in using `huggingface-cli login` in your terminal first!**

In [ ]:
#%pip install huggingface_hub -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Install the library if you haven't yet


from huggingface_hub import HfApi

api = HfApi()

HF_USERNAME = "nduka1999"

# 1. Upload Distilled RoBERTa
print("Uploading Distilled RoBERTa...")
api.upload_folder(
    folder_path="../models/roberta_distilled/best_model",
    repo_id=f"{HF_USERNAME}/cfpb-roberta-distilled",
    repo_type="model",
)

# 2. Upload ModernBERT
print("Uploading ModernBERT...")
api.upload_folder(
    folder_path="../models/modernbert_full/best_model",
    repo_id=f"{HF_USERNAME}/cfpb-modernbert",
    repo_type="model",
)

# 3. Upload Ensemble Artifacts (Stacker, KB, Bandit)
print("Uploading Ensemble Artifacts...")
api.upload_folder(
    folder_path="../models/ensemble",
    repo_id=f"{HF_USERNAME}/cfpb-ensemble-artifacts",
    repo_type="model",
)

print("✅ All models and artifacts successfully pushed to Hugging Face Hub!")



Uploading Distilled RoBERTa...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading ModernBERT...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading Ensemble Artifacts...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ All models and artifacts successfully pushed to Hugging Face Hub!
